# 03 · Partición definitiva y M-ComBat de referencia

No sortea cohortes. Ajusta M-ComBat en las cuatro cohortes de desarrollo y proyecta por separado las tres cohortes externas.

In [ ]:
from pathlib import Path
import os, sys

DATA_PREPARE = Path.cwd().resolve()
while not (DATA_PREPARE / "combat_ref.py").exists():
    if DATA_PREPARE.parent == DATA_PREPARE:
        raise FileNotFoundError("Ejecuta este notebook dentro de data_prepare/.")
    DATA_PREPARE = DATA_PREPARE.parent
os.chdir(DATA_PREPARE)
if str(DATA_PREPARE) not in sys.path:
    sys.path.insert(0, str(DATA_PREPARE))
print("DATA_PREPARE:", DATA_PREPARE)

In [ ]:
import json
import pandas as pd
from combat_ref import Config, cargar, fit_project_prepared, ReferenceMComBat

endpoint = json.loads((DATA_PREPARE / "output_prepared/endpoint.json").read_text())
cfg = Config(
    expr_path=str(DATA_PREPARE / "output_prepared/expr_prepared.csv"),
    meta_path=str(DATA_PREPARE / "output_prepared/metadata_prepared.csv"),
    active_labels=tuple(endpoint["active_labels"]),
    min_labels_per_cohort=int(endpoint["min_labels_per_cohort"]),
    out_dir=str(DATA_PREPARE / "output_mcombat"),
)
assert list(cfg.train_cohorts_fixed) == endpoint["train_cohorts"]
assert list(cfg.test_cohorts_fixed) == endpoint["test_cohorts"]
expr, meta = cargar(cfg)
res = fit_project_prepared(expr, meta, cfg)

In [ ]:
display(res["split"])
display(res["verificacion"])
assert res["verificacion"]["max_dif"].max() < 1e-9
print("TRAIN", res["expr_train"].shape, list(cfg.train_cohorts))
print("TEST ", res["expr_test"].shape, list(cfg.test_cohorts))

## Proyección posterior de una cohorte nueva

La cohorte debe tener al menos dos muestras, el mismo universo de genes y la misma convención logarítmica/pseudoconteo. No se recalcula el desarrollo.

In [ ]:
def proyectar_nueva_cohorte(expr_nueva, artifact_dir=DATA_PREPARE / "output_mcombat/artefacto_mcombat"):
    transformador = ReferenceMComBat.load(artifact_dir)
    return transformador.transform_cohorte(expr_nueva)

# Uso posterior (no se ejecuta aquí):
# expr_nueva_mcombat = proyectar_nueva_cohorte(expr_nueva_preparada)